# Source-only validation and protocol checks
Written only; no cells executed during creation. See README.md for run order.

In [ ]:
load_notebook(TASK2 / 'task2/evaluation/metrics.ipynb')

def validate_config(cfg, method):
    # Enforce the fixed comparison protocol rather than silently ignore YAML edits.
    recovery_keys = {'epochs', 'patience', 'learning_rate', 'batchnorm'} if cfg.get('experiment') == 'recovery_v1' else set()
    for key, value in CONFIG.items():
        if key in recovery_keys:
            continue
        if cfg.get(key) != value:
            raise ValueError(f'Configuration differs from shared ERM protocol: {key}')
    if not SPLIT_PATH.is_file():
        raise FileNotFoundError('Reuse the existing Task 2 split.')
    baseline = torch.load(BASELINE, map_location='cpu', weights_only=True)
    if baseline.get('method', 'source_only') != 'source_only':
        raise ValueError('Expected the Task 2 source-only checkpoint.')
    if baseline['config'] != CONFIG or baseline['split_sha256'] != split_hash(SPLIT_PATH):
        raise ValueError('ERM config/split mismatch.')
    if baseline['class_to_idx'] != {c: i for i,c in enumerate(CLASSES)}:
        raise ValueError('Class mapping mismatch.')
    if method == 'dan_dg' and (cfg['kernel_scales'] != [0.5, 1.0, 2.0] or cfg['lambda_dg'] not in [0.1, 1.0, 10.0]):
        raise ValueError('Use the fixed DAN-DG study.')
    if method == 'sam' and (cfg['rho'] != 0.05 or cfg['adaptive']):
        raise ValueError('Use standard SAM with rho=0.05.')
    del baseline
